<a href="https://colab.research.google.com/github/joyangelai/RAG/blob/main/RAGDoc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Document QA Pipeline (Retrieval- Augmented Generation)

Building a system that lets users upload complex multi page PDFs and chat with them using an LLM that only references the document facts to prevent hallucinations.

In [45]:
# Installation
!pip install -q langchain langchain-community chromadb openai pypdf tiktoken langchain-openai

In [46]:
!pip install -q sentence-transformers

In [47]:
# importing necesssary libraries
import os
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [48]:
# Step 1: Upload PDF file
print("Please upload your PDF file: ")

uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]




Please upload your PDF file: 


Saving bitcoin.pdf to bitcoin (1).pdf


In [49]:
# Step 2: Document ingestion
print(f"\n[1/3] Parsing {pdf_filename}.....")

loader = PyPDFLoader(pdf_filename)
docs = loader.load()



[1/3] Parsing bitcoin (1).pdf.....


In [50]:
from langchain_community.document_loaders.onedrive_file import CHUNK_SIZE
# Step 3: Strategic Text chunking (using an overlap so sentences cut off at the edge of a chunk retain context)

print("[2/3] Chunking text into contextual segments...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 150
)
chunks =  text_splitter.split_documents(docs)
print(f"Document successfully split into {len(chunks)} distinct chunks")


[2/3] Chunking text into contextual segments...
Document successfully split into 41 distinct chunks


In [51]:
# Step 4: Vector store & Embeddings Generation

print("[3/3] Generating Vector embeddings and indexing to Chromadb...")

embeddings = HuggingFaceEmbeddings(model_name= "all-MiniLM-L6-v2")
vector_stores = Chroma.from_documents(chunks,embeddings)

print("Vector database is live and fully indexed!!")

[3/3] Generating Vector embeddings and indexing to Chromadb...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector database is live and fully indexed!!


In [52]:
!pip install -q langchain-classic

In [53]:
# Section 2

from langchain_community.llms import HuggingFaceEndpoint
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate



In [54]:
# Step 1: Authenticate using HuggingFace Token
Hf_token = ""
os.environ["HUGGINGFACEHUB_API_TOKEN"] = Hf_token

In [55]:
# Step 2: Initialize free open source LLM (Mistral 7B)
llm = HuggingFaceEndpoint(
    repo_id = "mistralai/Mistral-7B-Instruct-v0.2",
    temperature= 0.1,
    max_new_tokens= 500
)

In [56]:
# Step 3: Connect free vector database retriever
retriever = vector_stores.as_retriever(search_kwargs={"k":3})

In [57]:
# Step 4: Strict system prompt to restrict hallucinations
system_prompt =  (
    "You are an advanced data science assistant. \n"
    "Use ONLY the following pieces of retrieved context to answer the question at the end. \n"
    "If you do not know the answer or if it's not explicitly found in text, say: \n"
    "'I cannot find the answer within a provided document.' Do not make things up. \n \n"
    "Context:\n{context}"
)

prompt_template =  ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [58]:
# Step 5: Assembling the RAG
question_answer_chain =  create_stuff_documents_chain(llm, prompt_template)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [59]:
# Step 6: Run Query (100% Free, Offline Local Model Version)
import torch
from transformers import pipeline

user_query = "What is the primary problem this document attempts to solve?"

if user_query:
    print(f" Searching local vector index for context...")
    # 1. Safely pull text chunks from your local vector store built
    docs = vector_stores.similarity_search(user_query, k=2) # k=2 to save local memory
    context = "\n\n".join([doc.page_content for doc in docs])

    print("Loading local offline LLM model into Colab memory (Qwen 0.5B)...")
    # 2. Set up a lightweight, completely local text generation pipeline
    pipe = pipeline(
        "text-generation",
        model="Qwen/Qwen2.5-0.5B-Instruct",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    # 3. Assemble the prompt instructions
    messages = [
        {"role": "system", "content": f"You are an advanced data science assistant. Use ONLY the following context to answer the question. If you do not know the answer, say 'I cannot find the answer within the provided document.'\n\nContext:\n{context}"},
        {"role": "user", "content": user_query}
    ]

    # 4. Generate the response text locally
    print("Generating answer locally...")
    outputs = pipe(messages, max_new_tokens=256, temperature=0.1, do_sample=False)

    print("\n ANSWER:")
    # Extract and clean up the model's text answer
    generated_text = outputs[0]["generated_text"][-1]["content"]
    print(generated_text)


 Searching local vector index for context...
Loading local offline LLM model into Colab memory (Qwen 0.5B)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Generating answer locally...

 ANSWER:
The primary problem this document attempts to solve is verifying the authenticity and integrity of transactions involving electronic coins, particularly those where one party may have attempted to double-spend the coin. This involves ensuring that only legitimate transactions are recorded on the blockchain, thereby preventing fraudulent activities such as double-spending.
